#### Imports

In [1]:
# Install required libraries (uncomment and run if needed)
!pip install stim pymatching numpy matplotlib torch tqdm networkx
!pip install torch_geometric

# For CUDA support with PyTorch Geometric, you may need:
# !pip install torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('..')

sys.path.insert(0, str(BASE_PATH))

# Common imports
import stim
import pymatching
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree
from datetime import datetime
from tqdm import tqdm
import networkx as nx
import os

# Import model classes and utilities from models.py
from models import (
    surface_code_circuit,
    count_logical_errors,
    ler_mwpm,
    plot_mwpm,
    SurfaceCodeSampler,
    SparseGraph,
    visualize_sparse_graph,
    GCNModel,
    GCN,
    DatasetCache,
)

C:\Users\Bill\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# If CUDA is available, show more details
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Current device: {torch.cuda.current_device()}")
else:
    print("PyTorch is using CPU only")

CUDA version: 13.0
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 5070 Ti
Current device: 0


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


#### Training

##### Definition

In [5]:
class GCNTrainer:
    """
    Highly customizable trainer for GCN models on quantum error correction data.
    
    This trainer handles the entire training pipeline:
    - Loads pre-generated graphs from cache OR generates them on-the-fly
    - Trains the GCN model
    - Returns the trained model
    
    Supports three modes:
    1. Pre-loaded graphs: Pass graphs directly via `graphs` parameter
    2. Cached datasets: Load from disk via `cache_name` parameter
    3. On-the-fly generation: Generate graphs during training (backwards compatible)
    
    Attributes:
        nickname (str): Human-readable name for the model
        distances (list[int]): Code distances to train on
        distance_weights (list[float]): Proportion of samples per distance
        p_values (list[float]): Physical error rates
        p_weights (list[float]): Distribution of error rates
        sample_size (int): Total number of training samples
        epochs (int): Number of training epochs
        batch_size (int): Training batch size
        lr (float): Learning rate
        hidden_dim (int): GCN hidden dimension
        num_layers (int): Number of GCN layers
        k_neighbors (int): K-neighbors for SparseGraph
        device (torch.device): Compute device
        graphs (list): Pre-loaded graphs (optional)
        cache_name (str): Name of cached dataset to load (optional)
    """
    
    def __init__(
        self,
        nickname: str,
        distances: list[int] = None,
        distance_weights: list[float] = None,
        p_values: list[float] = None,
        p_weights: list[float] = None,
        sample_size: int = None,
        epochs: int = 10,
        batch_size: int = 64,
        lr: float = 1e-3,
        hidden_dim: int = 128,
        num_layers: int = 4,
        k_neighbors: int = 6,
        device: torch.device = None,
        base_path: Path = None,
        graphs: list = None,
        cache_name: str = None
    ):
        """
        Initialize the GCN trainer.
        
        Args:
            nickname: Human-readable name for the model
            distances: Code distances to train on, e.g., [3, 5, 7] (not needed if using cache/graphs)
            distance_weights: Proportion for each distance, must sum to 1.0 (not needed if using cache/graphs)
            p_values: Physical error rates to sample from (not needed if using cache/graphs)
            p_weights: Distribution of error rates, must sum to 1.0 (not needed if using cache/graphs)
            sample_size: Total number of training samples to generate (not needed if using cache/graphs)
            epochs: Number of training epochs (default: 10)
            batch_size: Training batch size (default: 64)
            lr: Learning rate (default: 1e-3)
            hidden_dim: GCN hidden dimension (default: 128)
            num_layers: Number of GCN layers (default: 4)
            k_neighbors: K-neighbors for SparseGraph (default: 6)
            device: Compute device (defaults to CUDA if available)
            base_path: Base path for model/dataset storage (defaults to current directory)
            graphs: Pre-loaded list of PyG Data objects (optional, skips generation)
            cache_name: Name of cached dataset to load (optional, e.g., 'd5_baseline')
        
        Raises:
            ValueError: If using on-the-fly generation and required params are missing
            ValueError: If distances and distance_weights have different lengths
            ValueError: If p_values and p_weights have different lengths
        """
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.base_path = base_path
        
        # Store pre-loaded graphs or cache name
        self._graphs = graphs
        self._cache_name = cache_name
        
        # Determine data source mode
        if graphs is not None:
            self._mode = "preloaded"
            sample_size = len(graphs)
        elif cache_name is not None:
            self._mode = "cached"
            # We'll load the cache to get sample_size
            cache = DatasetCache(base_path=base_path, device=self.device)
            cache.load(cache_name)
            self._graphs = cache.get_graphs()
            sample_size = len(self._graphs)
        else:
            self._mode = "generate"
            # Validate generation parameters
            if distances is None or p_values is None or sample_size is None:
                raise ValueError("For on-the-fly generation, must provide distances, p_values, and sample_size")
            
            if distance_weights is None:
                distance_weights = [1.0 / len(distances)] * len(distances)
            if p_weights is None:
                p_weights = [1.0 / len(p_values)] * len(p_values)
            
            if len(distances) != len(distance_weights):
                raise ValueError(f"distances and distance_weights must have same length. "
                               f"Got {len(distances)} and {len(distance_weights)}")
            
            if len(p_values) != len(p_weights):
                raise ValueError(f"p_values and p_weights must have same length. "
                               f"Got {len(p_values)} and {len(p_weights)}")
            
            weight_sum = sum(distance_weights)
            if not np.isclose(weight_sum, 1.0, atol=1e-6):
                raise ValueError(f"distance_weights must sum to 1.0, got {weight_sum}")
            
            p_weight_sum = sum(p_weights)
            if not np.isclose(p_weight_sum, 1.0, atol=1e-6):
                raise ValueError(f"p_weights must sum to 1.0, got {p_weight_sum}")
        
        # Store configuration
        self.nickname = nickname
        self.distances = distances
        self.distance_weights = distance_weights
        self.p_values = p_values
        self.p_weights = p_weights
        self.sample_size = sample_size
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.k_neighbors = k_neighbors
        
        print(f"GCNTrainer initialized:")
        print(f"  Nickname: {nickname}")
        print(f"  Mode: {self._mode}")
        if self._mode == "cached":
            print(f"  Cache: {cache_name}")
        elif self._mode == "generate":
            print(f"  Distances: {distances} (weights: {distance_weights})")
            print(f"  Error rates: {p_values} (weights: {p_weights})")
        print(f"  Sample size: {sample_size:,}")
        print(f"  Epochs: {epochs}, Batch size: {batch_size}, LR: {lr}")
        print(f"  Model: hidden_dim={hidden_dim}, num_layers={num_layers}")
        print(f"  Device: {self.device}")
    
    def train(self, verbose: bool = True) -> GCN:
        """
        Execute the full training pipeline.
        
        This method:
        1. Uses pre-loaded/cached graphs OR generates them on-the-fly
        2. Shuffles the dataset
        3. Trains the GCN model
        4. Returns the trained model
        
        Args:
            verbose: Print progress information (default: True)
        
        Returns:
            GCN: The trained GCN model instance
        """
        import random
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Starting training pipeline for: {self.nickname}")
            print(f"{'='*60}")
        
        # Get graphs based on mode
        if self._mode in ("preloaded", "cached"):
            # Use pre-loaded or cached graphs
            all_graphs = self._graphs.copy()
            if verbose:
                print(f"\nUsing {self._mode} graphs: {len(all_graphs):,} samples")
        else:
            # Generate graphs on-the-fly (backwards compatible)
            if verbose:
                print(f"\nGenerating graphs on-the-fly...")
            
            sampler = SurfaceCodeSampler(p=self.p_values[0], device=self.device)
            graph_builder = SparseGraph(k_neighbors=self.k_neighbors, device=self.device)
            
            all_graphs = []
            
            for i, (d, weight) in enumerate(zip(self.distances, self.distance_weights)):
                # Calculate number of samples for this distance
                if i == len(self.distances) - 1:
                    n_samples = self.sample_size - len(all_graphs)
                else:
                    n_samples = int(round(self.sample_size * weight))
                
                if n_samples <= 0:
                    continue
                
                if verbose:
                    print(f"\nGenerating {n_samples:,} samples for distance d={d}...")
                
                detections, labels = sampler.sample(
                    d=d,
                    num_samples=n_samples,
                    p_values=self.p_values,
                    p_weights=self.p_weights
                )
                
                if verbose:
                    print(f"  Converting to sparse graphs...")
                
                graphs = []
                with tqdm(total=n_samples, desc=f"  Converting d={d}", 
                         unit="graph", leave=True, dynamic_ncols=True) as pbar:
                    for j in range(detections.shape[0]):
                        graph = graph_builder.to_pyg(detections[j], labels[j])
                        graphs.append(graph)
                        pbar.update(1)
                
                all_graphs.extend(graphs)
                
                if verbose:
                    print(f"  Generated {len(graphs):,} graphs (total: {len(all_graphs):,})")
        
        # Shuffle the dataset
        if verbose:
            print(f"\nShuffling {len(all_graphs):,} graphs...")
        random.shuffle(all_graphs)
        
        # Initialize the GCN model
        if verbose:
            print(f"\nInitializing GCN model...")
        
        gcn = GCN(
            nickname=self.nickname,
            in_channels=5,  # SparseGraph outputs 5 features
            hidden_dim=self.hidden_dim,
            num_layers=self.num_layers,
            device=self.device,
            base_path=self.base_path
        )
        
        # Train the model
        if verbose:
            print(f"\nStarting training...")
        
        gcn.train(
            graphs=all_graphs,
            epochs=self.epochs,
            batch_size=self.batch_size,
            lr=self.lr,
            verbose=verbose
        )
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Training complete for: {self.nickname}")
            print(f"{'='*60}")
        
        return gcn
    
    def __repr__(self) -> str:
        return (f"GCNTrainer(nickname='{self.nickname}', "
                f"distances={self.distances}, "
                f"sample_size={self.sample_size:,}, "
                f"epochs={self.epochs})")

##### Visualizing

In [6]:
# # Initialize sampler with default p=0.005
# sampler = SurfaceCodeSampler(p=0.005, device=device)

# # Define the distance to use for this training run
# d = 3

# # Define the error rates we want to sample from
# p_values = [0.001, 0.002, 0.003, 0.004, 0.005]

# # Evenly distributed weights (20% each)
# p_weights = [0.2, 0.2, 0.2, 0.2, 0.2]

# # Generate training data with 10,000 samples
# num_samples = 10000 #00
# train_detections, train_labels, p_indices = sampler.sample(
#     d=d,
#     num_samples=num_samples,
#     p_values=p_values,
#     p_weights=p_weights,
#     return_p_labels=True
# )

# # Print dataset info
# print(f"\nTraining data generated:")
# print(f"  Detections shape: {train_detections.shape}")
# print(f"  Labels shape: {train_labels.shape}")
# print(f"  Labels distribution: {train_labels.sum().item():.0f} positive / {num_samples} total")

# # Verify distribution of samples across error rates
# print(f"\nSamples per error rate:")
# for i, p in enumerate(p_values):
#     count = (p_indices == i).sum().item()
#     print(f"  p={p}: {count} samples ({count/num_samples*100:.1f}%)")

# # Generate graphs using SparseGraph
# graph_builder = SparseGraph(k_neighbors=6, device=device)
# graphs = graph_builder.batch_to_pyg(train_detections, train_labels)

# # Find the first non-empty graph for visualization
# for i, g in enumerate(graphs):
#     if g.x.shape[0] > 0:
#         print(f"Visualizing graph from sample {i}")
#         visualize_sparse_graph(g, title=f"Sample {i} - Sparse Graph")
#         break

##### Logic

In [7]:
# Directory where models are saved
models_dir = BASE_PATH / "models" / "gcn"

# Distances to train
distances = [3, 5, 7, 9]

# Store trained models
trained_models = {}

for d in distances:
    nickname = f"d{d}_baseline"
    
    # Check if model already exists (any file starting with nickname_)
    existing = list(models_dir.glob(f"{nickname}_*.pt"))
    if existing:
        print(f"Model '{nickname}' already exists: {existing[0].name}, skipping...")
        continue
    
    print(f"\n{'='*60}")
    print(f"Training {nickname}...")
    print(f"{'='*60}")
    
    trainer = GCNTrainer(
        nickname=nickname,
        distances=[d],
        distance_weights=[1.0],
        p_values=[0.001, 0.003, 0.005, 0.007],
        p_weights=[0.25, 0.25, 0.25, 0.25],
        sample_size=10**6,
        epochs=10,
        batch_size=128,
        lr=1e-3,
        hidden_dim=128,
        num_layers=4,
        base_path=BASE_PATH
    )
    
    trained_models[nickname] = trainer.train()

# Save all trained models
print(f"\n{'='*60}")
print(f"Saving {len(trained_models)} trained models...")
print(f"{'='*60}")

for nickname, model in trained_models.items():
    model.save(nickname)

print(f"\nDone! Trained and saved {len(trained_models)} models.")

Model 'd3_baseline' already exists: d3_baseline_2026-01-12_21-26-00.pt, skipping...

Training d5_baseline...
GCNTrainer initialized:
  Nickname: d5_baseline
  Distances: [5] (weights: [1.0])
  Error rates: [0.001, 0.003, 0.005, 0.007] (weights: [0.25, 0.25, 0.25, 0.25])
  Sample size: 1,000,000
  Epochs: 10, Batch size: 128, LR: 0.001
  Model: hidden_dim=128, num_layers=4
  Device: cuda

Starting training pipeline for: d5_baseline
SurfaceCodeSampler initialized:
  Default error rate: 0.001
  Device: cuda
  Mode: Dynamic (supports any code distance)
SparseGraph initialized:
  K neighbors: 6
  Device: cuda
  Mode: Dynamic (supports any code distance)

Generating 1,000,000 samples for distance d=5...
  Converting to sparse graphs...


  Converting d=5: 100%|██████████| 1000000/1000000 [28:55<00:00, 576.11graph/s]


  Generated 1,000,000 graphs (total: 1,000,000)

Shuffling 1,000,000 graphs...

Initializing GCN model...
GCN initialized: GCN(nickname='d5_baseline', in_channels=5, hidden_dim=128, num_layers=4)

Starting training...

Training: GCN(nickname='d5_baseline', in_channels=5, hidden_dim=128, num_layers=4)
Epochs: 10 | Batch size: 128 | LR: 0.001
Training samples: 1000000


Training: 100%|██████████| 78130/78130 [1:40:13<00:00, 12.99it/s, epoch=10/10, loss=0.2292, acc=88.5%]



Training complete! Final - Loss: 0.2417, Accuracy: 88.3%

Training complete for: d5_baseline

Training d7_baseline...
GCNTrainer initialized:
  Nickname: d7_baseline
  Distances: [7] (weights: [1.0])
  Error rates: [0.001, 0.003, 0.005, 0.007] (weights: [0.25, 0.25, 0.25, 0.25])
  Sample size: 1,000,000
  Epochs: 10, Batch size: 128, LR: 0.001
  Model: hidden_dim=128, num_layers=4
  Device: cuda

Starting training pipeline for: d7_baseline
SurfaceCodeSampler initialized:
  Default error rate: 0.001
  Device: cuda
  Mode: Dynamic (supports any code distance)
SparseGraph initialized:
  K neighbors: 6
  Device: cuda
  Mode: Dynamic (supports any code distance)

Generating 1,000,000 samples for distance d=7...
  Converting to sparse graphs...


...

  Generated 1,000,000 graphs (total: 1,000,000)

Shuffling 1,000,000 graphs...

Initializing GCN model...
GCN initialized: GCN(nickname='d7_baseline', in_channels=5, hidden_dim=128, num_layers=4)

Starting training...

Training: GCN(nickname='d7_baseline', in_channels=5, hidden_dim=128, num_layers=4)
Epochs: 10 | Batch size: 128 | LR: 0.001
Training samples: 1000000


Training: 100%|██████████| 78130/78130 [1:47:36<00:00, 12.10it/s, epoch=10/10, loss=0.3739, acc=80.7%]



Training complete! Final - Loss: 0.3801, Accuracy: 79.7%

Training complete for: d7_baseline

Training d9_baseline...
GCNTrainer initialized:
  Nickname: d9_baseline
  Distances: [9] (weights: [1.0])
  Error rates: [0.001, 0.003, 0.005, 0.007] (weights: [0.25, 0.25, 0.25, 0.25])
  Sample size: 1,000,000
  Epochs: 10, Batch size: 128, LR: 0.001
  Model: hidden_dim=128, num_layers=4
  Device: cuda

Starting training pipeline for: d9_baseline
SurfaceCodeSampler initialized:
  Default error rate: 0.001
  Device: cuda
  Mode: Dynamic (supports any code distance)
SparseGraph initialized:
  K neighbors: 6
  Device: cuda
  Mode: Dynamic (supports any code distance)

Generating 1,000,000 samples for distance d=9...
  Converting to sparse graphs...


  Converting d=9:   9%|▉         | 93412/1000000 [1:34:33<10:10:29, 24.75graph/s]  Converting d=9:   9%|▉         | 93821/1000000 [1:35:06<19:02:06, 13.22graph/s]  Converting d=9:   9%|▉         | 94102/1000000 [1:35:29<19:27:20, 12.93graph/s]  Converting d=9:  10%|█         | 102064/1000000 [1:42:42<15:03:37, 16.56graph/s]


KeyboardInterrupt: 